<a href="https://colab.research.google.com/github/CevdetSatarr/FlyRank-intern/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CevdetSatarr/FlyRank-intern/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

**Lane 2, continued from ML-08.** This notebook applies the same careful reading used on FlyRank's own research paper to my own Week-5 model.

> Read `skills/README.md`, then load `hunting-leakage-and-validating` + `flyrank/flyrank-data` before working this notebook.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1 — “What Predicts Health?” (Random Forest feature importance).** The paper reports Average Position (43%) and Impressions (32%) as the top predictors of Health Score, described as “holdout-tested.”

*My methodology question:* Health Score is itself defined as Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts) — the paper states this plainly in its own methodology section. If two of the four ingredients of the label are also the model's top two “predictors,” is the model discovering a real driver of health, or is it recovering its own scoring formula? The paper's own caveat (“importance is descriptive rather than causal”) gestures at this, but doesn't fully resolve it: a holdout split protects against overfitting noise, it doesn't protect against a target that's partly arithmetic on the inputs. A cleaner test would predict something structurally independent of Health Score's formula — e.g. future click growth — to see whether Position and Impressions still dominate once the label can't be partly reconstructed from them directly.

**Finding 2 — “What Predicts Growth?” (Logistic Regression, 71% holdout accuracy).** The paper reports 71% holdout accuracy separating growing from declining pages.

*My methodology question:* Finding #1 elsewhere in the same paper reports 74,835 growing pages versus 45,619 declining ones — a roughly 62/38 base rate. Against that, is 71% accuracy nine points of real separating power, or closer to the floor a naive “always predict growing” rule would already clear? The paper doesn't state the majority-class baseline next to this number, so a reader has no way to tell from the page itself how much the model is actually adding. This is the exact base-rate discipline the `hunting-leakage-and-validating` skill asks for: every score reported next to its naive baseline, always.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Week-5 already used a grouped split (by `client_hash_id`), so the “before” here is reconstructed deliberately: a naive random row-level split on the exact same data and features, to show what the grouped split actually bought me. Same Random Forest, same features, same base frame.

In [8]:
%pip -q install duckdb scikit-learn
import os, duckdb, pandas as pd, numpy as np

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

base = con.sql("""
    SELECT
        f.client_hash_id, f.content_hash_id,
        SUM(f.gsc_impressions) AS gsc_impressions,
        AVG(f.gsc_avg_position) AS gsc_avg_position,
        ANY_VALUE(DATE_DIFF('day', c.content_created_date, DATE '2026-03-31')) AS content_age_days,
        SUM(CASE WHEN EXTRACT(DAY FROM f.report_date) <= 15 THEN f.gsc_clicks ELSE 0 END) AS first_half_clicks,
        SUM(CASE WHEN EXTRACT(DAY FROM f.report_date) > 15  THEN f.gsc_clicks ELSE 0 END) AS second_half_clicks
    FROM {FACT} f LEFT JOIN {DIM_CONTENT} c ON c.content_hash_id = f.content_hash_id
    GROUP BY 1, 2 HAVING SUM(f.gsc_impressions) >= 100
""".format(FACT=FACT, DIM_CONTENT=DIM_CONTENT)).df()

base = base.dropna(subset=['content_age_days'])
base = base[base['first_half_clicks'] > 0].copy()
base['click_trend_pct'] = (base['second_half_clicks'] - base['first_half_clicks']) / base['first_half_clicks'] * 100
base['is_declining'] = (base['click_trend_pct'] < 0).astype(int)

honest_features = ['gsc_impressions', 'gsc_avg_position', 'content_age_days']
X, y, groups = base[honest_features], base['is_declining'], base['client_hash_id']
print(f'{len(base):,} rows, {groups.nunique()} clients, population decline rate: {y.mean():.3f}')


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

49,396 rows, 35 clients, population decline rate: 0.543


In [9]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

def precision_at_k(y_true, y_score, k=20):
    order = np.argsort(-y_score)[:k]
    return y_true.iloc[order].mean()

def evaluate(X_train, X_test, y_train, y_test, label):
    rf = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42, n_jobs=-1).fit(X_train, y_train)
    proba = rf.predict_proba(X_test)[:, 1]
    preds = (proba >= 0.5).astype(int)
    return {
        'split': label,
        'test_rows': len(X_test),
        'test_decline_rate': y_test.mean(),
        'accuracy': accuracy_score(y_test, preds),
        'precision': precision_score(y_test, preds, zero_division=0),
        'recall': recall_score(y_test, preds, zero_division=0),
        'precision_at_20': precision_at_k(y_test, proba, 20),
    }

results = []

# BEFORE: naive random row-level split (what Week-4 originally did)
Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(X, y, test_size=0.25, random_state=42)
results.append(evaluate(Xtr_r, Xte_r, ytr_r, yte_r, 'BEFORE: random row split'))

# AFTER: grouped split by client (Week-5)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
Xtr_g, Xte_g = X.iloc[train_idx], X.iloc[test_idx]
ytr_g, yte_g = y.iloc[train_idx], y.iloc[test_idx]
results.append(evaluate(Xtr_g, Xte_g, ytr_g, yte_g, 'AFTER: grouped split (by client)'))

comparison = pd.DataFrame(results).round(3)
comparison


,split,test_rows,test_decline_rate,accuracy,precision,recall,precision_at_20
0,BEFORE: random row split,12349,0.542,0.597,0.600,0.772,0.85
1,AFTER: grouped split (by client),1213,0.575,0.613,0.605,0.944,0.90


In this run, the grouped split actually scored higher on accuracy and recall than the random split — the opposite of what memorization-inflation would predict. This isn't evidence that grouping improved the model; with only ~35 clients, the grouped test set (1,213 rows) is small enough that its composition — which specific client(s) landed in test, and their above-population decline rate (0.575 vs 0.542) — likely drove the difference more than genuine generalization did. The honest conclusion here isn't 'grouping revealed inflation,' it's that a single grouped split with this few clients is too unstable to trust the direction of the gap at all — which is exactly why GroupKFold averaged over multiple folds would be the right next step, not a single split either way.

**Reading the before/after:** if the random-split ('before') numbers look noticeably stronger than the grouped-split ('after') numbers, that gap is itself the finding — it's roughly how much of the random-split score was the model memorizing which client a row belonged to, rather than a pattern that generalizes to a client it hasn't seen. If the two are close, that's also worth stating plainly, since it means client identity wasn't doing much work either way in this lane.

**A second, more specific finding from the grouped split itself:** with only ~35 clients total, a 25%-of-clients split doesn't reliably produce a 25%-of-rows test set — client size varies a lot, so the grouped test set can end up small and its decline rate can drift from the population rate. That's not leakage; it's a validation-design weakness worth naming directly rather than treating the grouped-split number as automatically trustworthy just because it's grouped. The fix for a future pass would be `GroupKFold` averaged over several folds instead of one single group split, so no single unlucky partition decides the reported number.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Running the toolkit's attack checklist against the final three-feature set:

- **`gsc_impressions`** — this month's total Search Console impressions. Static once the month closes; not derived from `is_declining`. Not a sibling of the label. **Clean.**
- **`gsc_avg_position`** — this month's average rank. Same reasoning — an input to search performance, not a function of the click-trend label. **Clean.**
- **`content_age_days`** — static content metadata (`content_created_date`), fixed regardless of this month's outcome. **Clean.**
- **Explicitly excluded:** `gsc_clicks` and `second_half_clicks` — these are direct ingredients of `is_declining` itself (the label is built from first-half vs. second-half clicks). Week-4's deliberate leak test confirmed this: adding `second_half_clicks` as a “feature” pushed accuracy toward a near-perfect, meaningless number, which is exactly the leakage taxonomy's label-derived-feature symptom (one feature towers over the others, score jumps toward 1.0).
- **No product/decision flags used** — no existing FlyRank optimization flag or prior score is fed in as a feature; the only “baseline to beat” role a rule-based score plays is the Week-4 hand-crafted rule itself, kept separate as a comparison point, never as an input.
- **Population check:** rows are included only if `gsc_impressions >= 100` for the month — that threshold is decided from data available at scoring time, not from anything in the outcome window, so it isn't smuggling future information into which rows even get considered.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (Week-5 read of the comparison table):** “Random Forest is stronger than Logistic Regression because it captures non-linear interactions a linear model can't.”

**Rewritten, against the real numbers:** On the grouped-split test set (1,213 rows), Random Forest showed a modest, directional accuracy improvement over Logistic Regression (61.5% vs. 57.5%, base rate 57.5%) — but the two models were **identical on Precision@20 (0.900 each)**, which is the metric that actually matches how Lane 2's output gets used (a reviewer working top-down through a ranked queue). On this evidence, Random Forest's added complexity did not produce a measured advantage on the metric that matters most for this lane's real use case, even though it edged out accuracy on a small, single-split test set. Any claim that Random Forest is the “better” model here should be treated as directional and split-specific, not settled — both models are decision-support inputs to the same ranked review queue, not a finished product.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ x] Every section above is filled — markdown thinking AND the code that backs it
- [ x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ x] No client names, URLs, or private queries anywhere
- [ x] My claims use careful words: observed, measured, directional, decision-support
- [ x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.